In [1]:
import tempfile
import pickle
import sys
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint

from one.api import ONE
from scipy.special import logit, softmax
import os
from os.path import join
import pickle as pkl
from brainwidemap.bwm_loading import bwm_query, bwm_units, load_trials_and_mask, merge_probes

import concurrent.futures
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import sys

from one.api import ONE
from brainbox.io.one import SessionLoader
from iblatlas.atlas import BrainRegions
from sklearn.metrics import balanced_accuracy_score

from brainwidemap.bwm_loading import (
    bwm_query,
    load_good_units,
    load_trials_and_mask,
    merge_probes,
    bwm_units,
)
from manifold.decoding.functions.utils import check_config_decoding

config = check_config_decoding()
MY_REGIONS = config["prior_regions"]

In [3]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)
print("Querying BWM Units...")

units_df = bwm_units(one)
flattened_regions = [item for sublist in MY_REGIONS for item in sublist]
relevant_pids = units_df[units_df["Beryl"].isin(flattened_regions)]["pid"].unique()

bwm_df = bwm_query(one)
# change subset df: use all valid eids
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]
list_of_eids = subset_df["eid"].unique()

Querying BWM Units...
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [2]:
from manifold.decoding.functions.nulldistributions import generate_null_distribution_session

In [3]:
from manifold.decoding.functions.utils import find_incongruent_trials

In [4]:
one = ONE(
    # base_url="https://openalyx.internationalbrainlab.org",
    # password="international",
    # silent=True,
    # username="intbrainlab",
    mode="local",
)
session_id = list_of_eids[0]
# try to make it local
session_data = bwm_df[bwm_df["eid"] == session_id]
pids = session_data["pid"].tolist()
probes = session_data["probe_name"].tolist()

trials, mask = load_trials_and_mask(
    one,
    session_id,
    exclude_nochoice=False,
    exclude_unbiased=True,  # should include no-choice trials
    min_rt=None,
    max_rt=None,
)

NameError: name 'list_of_eids' is not defined

In [51]:
target = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_real.npy"
)

In [52]:
target.shape

(446, 1)

In [53]:
pseudo = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_pseudo.npy"
)

In [54]:
pseudo.shape

(4, 446, 1)

In [49]:
pseudo_congruence_data = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/congruence_pseudo.npy"
)

In [55]:
pseudo_congruence_data.shape

(5, 2, 565)

In [56]:
trials_df = pd.read_parquet(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/trials.pqt"
)

In [57]:
congruence_mask, incongruence_mask = pseudo_congruence_data[0, :]

In [58]:
congruence_rt_mask = congruence_mask[trials_df["mask"]]
incongruence_rt_mask = incongruence_mask[trials_df["mask"]]

In [59]:
target_real = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_real.npy"
)
target_pseudo = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_pseudo.npy"
)
region_real_predictions = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/LGd_predictions_real.npy"
)
region_pseudo_predictions = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/LGd_predictions_pseudo.npy"
)

In [60]:
predictions = region_real_predictions[0, :]

In [61]:
predicted_labels = np.where(predictions >= 0.5, 1, -1)

In [62]:
balanced_accuracy_score(predicted_labels[congruence_rt_mask], target_real[congruence_rt_mask])

0.5068732090203081

In [63]:
balanced_accuracy_score(predicted_labels[incongruence_rt_mask], target_real[incongruence_rt_mask])

0.4951923076923077

In [67]:
summary_df = pd.read_parquet(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/MG_decoding_summary.pqt"
)

In [68]:
summary_df

,eid,condition,subject,probe,region,pseudo_id,run_id,N_units,R2_test,nb_trials,weights,intercept
0,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,-1,0,12,0.484901,446,"[0.015563884190923255, 0.04702433194409679, -0...",-0.441876
1,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,-1,1,12,0.445677,446,"[0.02912881408092697, 0.03523567022393206, -0....",-0.739650
2,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,1,0,12,0.535435,446,"[0.0031150540201239905, 0.003030101860560913, ...",0.297589
3,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,1,1,12,0.468337,446,"[0.005774653317218234, 0.003166913048864252, -...",0.145840
4,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,2,0,12,0.521432,446,"[-0.01021643245175739, 0.013375500258867103, -...",0.212460
5,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,2,1,12,0.468054,446,"[-0.01239426142806618, 0.04401889174854468, -0...",0.206489
6,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,3,0,12,0.487628,446,"[-0.009590028058817684, -0.006919660325824595,...",0.229617
7,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,3,1,12,0.553563,446,"[-0.03681359499301835, -0.006684752955067372, ...",0.845278
8,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,4,0,12,0.513643,446,"[-0.00278622474285558, 0.029220862661522785, -...",0.062487
9,6713a4a7-faed-4df2-acab-ee4e63326f8d,all,NYU-11,probe00,MG,4,1,12,0.510012,446,"[-0.005738791729941859, 0.024386580230724052, ...",0.132673


In [ ]:
summary_df_alt = pd.read_parquet